In [1]:
import numpy as np
from h5py import File
from matplotlib import pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader,Dataset
filename = 'feature.h5'

In [29]:
with File(filename,'r') as h5:
    print(h5.keys())
    
    src_label = h5['src_label'][:]
    src_feature = h5['src_feature'][:]

    
    tgt_label = h5['tgt_label'][:]
    tgt_feature = h5['tgt_feature'][:]
    
    tgt_feature_wo = h5['tgt_feature_wo'][:]
    tgt_label_wo = h5['tgt_label_wo'][:]

<KeysViewHDF5 ['src_feature', 'src_label', 'tgt_feature', 'tgt_feature_wo', 'tgt_label', 'tgt_label_wo']>


In [7]:
def scl(p):
    p = np.clip(p,1e-8,1)
    p_mean = p.mean(axis=0)
    return -np.mean(p*np.log(p))+np.mean(p_mean*np.log(p_mean))
def scl_y(y):
   y_max = y.max(axis=-1,keepdims=True)
   dy = y_max-y
   return -dy.mean()
def scl_torch(p):
    p = torch.clip(p,1e-8,1)
    p_mean = p.mean(dim=0)
    return -torch.mean(p*torch.log(p))+torch.mean(p_mean*torch.log(p_mean))*10
def scl_torch_y(y):
   y_max = y.max(dim=-1,keepdims=True).values
   dy = (y_max-y)**2
   return -dy.mean()
def top1(y_P,label):
    label_P = y_P.argmax(axis=-1)
    r = np.where(label_P==label,1,0)
    return r.mean()

In [4]:
device = 'cuda:1'
class DY(torch.nn.Module):
    def __init__(self,class_num,feature_num):
        super().__init__()
        self. dy = torch.nn.Parameter(torch.zeros(class_num))
        self.M = torch.nn.Parameter(torch.zeros((class_num,feature_num)))
    def forward(self,f):
        return self.dy.unsqueeze(0)+torch.einsum('bf,cf->bc',f,self.M)

In [30]:
src_label_torch = torch.from_numpy(src_label).to(device)
src_feature_torch = torch.from_numpy(src_feature).to(device)
tgt_label_torch = torch.from_numpy(tgt_label).to(device)
tgt_feature_torch = torch.from_numpy(tgt_feature).to(device)
tgt_label_wo_torch = torch.from_numpy(tgt_label_wo).to(device)
tgt_feature_wo_torch = torch.from_numpy(tgt_feature_wo).to(device)

In [ ]:

dy = DY(31,2048)
dy.to(device)

w = 0.01
dy.train()
opt = torch.optim.SGD(dy.parameters(),lr=1e-2)
sclL = []
tgt_acL =[]
for i in range(10000):
    dy.zero_grad()
    tgt_y_torch_dy  = dy(tgt_feature_torch )
    tgt_y_torch_dy_p = torch.nn.functional.softmax(tgt_y_torch_dy,dim=-1)
    
    
    loss_sur = torch.nn.functional.cross_entropy(tgt_y_torch_dy,tgt_label_torch)
    loss = loss_sur

    loss.backward()
    opt.step()
    tgt_y_torch_dy_num = tgt_y_torch_dy.detach().cpu().numpy()
    tgt_ac = top1(tgt_y_torch_dy_num,tgt_label)
    if i%10==0:
        print(tgt_ac)
        #sclL.append(loss_scl.item())
        tgt_acL.append(tgt_ac)

0.032670454545454544
0.6814630681818182
0.7269176136363636
0.7503551136363636
0.7610085227272727
0.7727272727272727
0.7830255681818182
0.7926136363636364
0.7972301136363636
0.7993607954545454
0.8036221590909091
0.8075284090909091
0.8139204545454546
0.8178267045454546
0.8199573863636364
0.8238636363636364
0.8259943181818182
0.8270596590909091
0.8302556818181818
0.8313210227272727
0.8341619318181818
0.8345170454545454
0.8348721590909091
0.8355823863636364
0.8370028409090909
0.8373579545454546
0.8384232954545454
0.83984375
0.8405539772727273
0.8419744318181818
0.8430397727272727
0.84375
0.8451704545454546
0.8458806818181818
0.8462357954545454
0.8473011363636364
0.8487215909090909
0.8497869318181818
0.8497869318181818
0.8508522727272727
0.8512073863636364
0.8529829545454546
0.8536931818181818
0.8536931818181818
0.8583096590909091
0.8586647727272727
0.8597301136363636
0.8607954545454546
0.8618607954545454
0.86328125
0.8639914772727273
0.8647017045454546
0.8654119318181818
0.8664772727272727

In [20]:
dy = DY(31,2048)
dy.to(device)

w = 0.01
dy.train()
opt = torch.optim.SGD(dy.parameters(),lr=1e-2)
opt= torch.optim.AdamW(dy.parameters(),lr=1e-3)
sclL = []
tgt_acL =[]
T =1
for i in range(20000):
    dy.zero_grad()
    tgt_y_torch_dy  = dy(tgt_feature_torch )
    tgt_y_torch_dy_p = torch.nn.functional.softmax(tgt_y_torch_dy/T,dim=-1)
    
    src_y_torch_dy  = dy(src_feature_torch )
    src_y_torch_dy_p = torch.nn.functional.softmax(src_y_torch_dy/T,dim=-1)
    
    tgt_y_torch_dy_wo  = dy(tgt_feature_wo_torch )
    tgt_y_torch_dy_wo_p = torch.nn.functional.softmax(tgt_y_torch_dy_wo/T,dim=-1)
    
    loss_sur = torch.nn.functional.cross_entropy(src_y_torch_dy,src_label_torch)
    loss_scl = scl_torch(tgt_y_torch_dy_p)
    #loss_sur = torch.nn.functional.cross_entropy(tgt_y_torch_dy,tgt_label_torch)
    loss =loss_sur#+(1-alpha)*loss_scl

    loss.backward()
    opt.step()
    tgt_y_torch_dy_num = tgt_y_torch_dy.detach().cpu().numpy()
    tgt_ac = top1(tgt_y_torch_dy_num,tgt_label)
    
    tgt_y_torch_dy_wo_num = tgt_y_torch_dy_wo.detach().cpu().numpy()
    tgt_ac_wo = top1(tgt_y_torch_dy_wo_num,tgt_label_wo)
    if i%10==0:
        print(tgt_ac,tgt_ac_wo)
        #sclL.append(loss_scl.item())
        tgt_acL.append(tgt_ac)

0.032670454545454544 0.032670454545454544
0.6263316761363636 0.6331676136363636
0.6482599431818182 0.6580255681818182
0.6556285511363636 0.6626420454545454
0.6508345170454546 0.6615767045454546
0.6520774147727273 0.6629971590909091
0.6518110795454546 0.6626420454545454
0.6520774147727273 0.6608664772727273
0.6507457386363636 0.66015625
0.6509232954545454 0.6598011363636364
0.650390625 0.6590909090909091
0.650390625 0.65625
0.650390625 0.6558948863636364
0.6498579545454546 0.6555397727272727
0.6498579545454546 0.6548295454545454
0.6497691761363636 0.6548295454545454
0.6500355113636364 0.6544744318181818
0.6500355113636364 0.6544744318181818
0.6495028409090909 0.6537642045454546
0.6493252840909091 0.6534090909090909
0.6495028409090909 0.6534090909090909
0.6493252840909091 0.6534090909090909
0.6488813920454546 0.6534090909090909
0.6486150568181818 0.6534090909090909
0.6485262784090909 0.6534090909090909
0.6482599431818182 0.6530539772727273
0.6479936079545454 0.6530539772727273
0.64790482

KeyboardInterrupt: 

In [32]:
dy = DY(31,2048)
dy.to(device)

w = 0.01
dy.train()
opt = torch.optim.SGD(dy.parameters(),lr=1e-2)
#opt= torch.optim.AdamW(dy.parameters(),lr=1e-3)
sclL = []
tgt_acL =[]
T_src = 1
T_tgt = 1
for i in range(20000):
    alpha = torch.exp(torch.tensor(-i/1000)).to(device)*0.999+0.001
    dy.zero_grad()
    tgt_y_torch_dy  = dy(tgt_feature_torch )
    tgt_y_torch_dy_p = torch.nn.functional.softmax(tgt_y_torch_dy/T_tgt,dim=-1)
    
    src_y_torch_dy  = dy(src_feature_torch )
    src_y_torch_dy_p = torch.nn.functional.softmax(src_y_torch_dy/T_src,dim=-1)
    
    tgt_y_torch_dy_wo  = dy(tgt_feature_wo_torch )
    tgt_y_torch_dy_wo_p = torch.nn.functional.softmax(tgt_y_torch_dy_wo/T_tgt,dim=-1)
    
    loss_sur = torch.nn.functional.cross_entropy(src_y_torch_dy,src_label_torch)
    loss_scl = scl_torch(tgt_y_torch_dy_p)
    #loss_sur = torch.nn.functional.cross_entropy(tgt_y_torch_dy,tgt_label_torch)
    loss = alpha*loss_sur+(1-alpha)*loss_scl

    loss.backward()
    opt.step()
    tgt_y_torch_dy_num = tgt_y_torch_dy.detach().cpu().numpy()
    tgt_ac = top1(tgt_y_torch_dy_num,tgt_label)
    tgt_y_torch_dy_wo_num = tgt_y_torch_dy_wo.detach().cpu().numpy()
    tgt_ac_wo = top1(tgt_y_torch_dy_wo_num,tgt_label_wo)
    src_y_torch_dy_num = src_y_torch_dy.detach().cpu().numpy()
    src_ac = top1(src_y_torch_dy_num,src_label)
    if i%10==0:
        print(tgt_ac,tgt_ac_wo,src_ac,alpha)
        #sclL.append(loss_scl.item())
        tgt_acL.append(tgt_ac)

0.032670454545454544 0.032670454545454544 0.02388392857142857 tensor(1., device='cuda:1')
0.20649857954545456 0.20099431818181818 0.46450892857142856 tensor(0.9901, device='cuda:1')
0.30685369318181815 0.3025568181818182 0.6863839285714286 tensor(0.9802, device='cuda:1')
0.3891690340909091 0.3895596590909091 0.7796875 tensor(0.9705, device='cuda:1')
0.4511363636363636 0.4520596590909091 0.8450892857142858 tensor(0.9608, device='cuda:1')
0.49122869318181817 0.49360795454545453 0.8785714285714286 tensor(0.9513, device='cuda:1')
0.5243252840909091 0.5262784090909091 0.9066964285714286 tensor(0.9418, device='cuda:1')
0.5518465909090909 0.5511363636363636 0.9287946428571429 tensor(0.9325, device='cuda:1')
0.5760653409090909 0.5717329545454546 0.9395089285714285 tensor(0.9232, device='cuda:1')
0.5948508522727273 0.5905539772727273 0.9506696428571428 tensor(0.9140, device='cuda:1')
0.6121448863636364 0.6100852272727273 0.9578125 tensor(0.9049, device='cuda:1')
0.6248224431818182 0.625 0.96294

In [15]:
dy = DY(31,2048)
dy.to(device)

w = 0.01
dy.train()
opt = torch.optim.SGD(dy.parameters(),lr=1e-2)
#opt= torch.optim.AdamW(dy.parameters(),lr=1e-3)
sclL = []
tgt_acL =[]
T = 10
for i in range(20000):
    alpha = torch.exp(torch.tensor(-i/2000)).to(device)*0.95+0.05
    alpha = 0.5
    dy.zero_grad()
    tgt_y_torch_dy  = dy(tgt_feature_torch )
    tgt_y_torch_dy_p = torch.nn.functional.softmax(tgt_y_torch_dy/T,dim=-1)
    
    src_y_torch_dy  = dy(src_feature_torch )
    src_y_torch_dy_p = torch.nn.functional.softmax(src_y_torch_dy/T,dim=-1)
    
    loss_sur = torch.nn.functional.cross_entropy(src_y_torch_dy,src_label_torch)
    loss_scl = scl_torch(tgt_y_torch_dy_p)
    loss_sur = torch.nn.functional.cross_entropy(tgt_y_torch_dy,tgt_label_torch)
    loss_scl = scl_torch(src_y_torch_dy_p)
    loss = alpha*loss_sur+(1-alpha)*loss_scl

    loss.backward()
    opt.step()
    tgt_y_torch_dy_num = tgt_y_torch_dy.detach().cpu().numpy()
    tgt_ac = top1(tgt_y_torch_dy_num,tgt_label)
    src_y_torch_dy_num = src_y_torch_dy.detach().cpu().numpy()
    src_ac = top1(src_y_torch_dy_num,src_label)
    if i%10==0:
        print(tgt_ac,src_ac)
        #sclL.append(loss_scl.item())
        #tgt_acL.append(tgt_ac)

0.032670454545454544 0.0234375
0.6360085227272727 0.5234375
0.6839488636363636 0.6067708333333334
0.7102272727272727 0.65625
0.7290482954545454 0.671875
0.7407670454545454 0.6744791666666666
0.7503551136363636 0.6848958333333334
0.7560369318181818 0.6848958333333334
0.7610085227272727 0.6901041666666666
0.7652698863636364 0.6927083333333334
0.7727272727272727 0.7005208333333334
0.7787642045454546 0.7109375
0.7830255681818182 0.71875
0.7879971590909091 0.7239583333333334
0.7922585227272727 0.7239583333333334
0.7943892045454546 0.7369791666666666
0.7972301136363636 0.7447916666666666
0.7979403409090909 0.7447916666666666
0.7993607954545454 0.7421875
0.8004261363636364 0.7421875
0.8036221590909091 0.7447916666666666
0.8057528409090909 0.7473958333333334
0.8075284090909091 0.75
0.8117897727272727 0.7526041666666666
0.8139204545454546 0.7526041666666666
0.81640625 0.75
0.8178267045454546 0.75
0.8192471590909091 0.7526041666666666
0.8196022727272727 0.7578125
0.8217329545454546 0.76041666666